# Tehnici de reprezentare și preprocesare în NLP

Cheatsheet didactic: exemple simple, pas cu pas, cu **pandas / scikit-learn** și **Python clasic / NLTK**.

| Secțiune | Librării |
|---|---|
| One-Hot Encoding, Bag of Words, TF-IDF | `pandas`, `sklearn` |
| Tokenization → Parsing | Python clasic + `nltk` |

## 0. Setup

Rulează o dată celula de mai jos (descarcă resursele NLTK necesare).

In [1]:
import re
import string
from collections import Counter

import pandas as pd
import nltk
from nltk.corpus import stopwords
from nltk.stem import PorterStemmer, WordNetLemmatizer
from nltk.tokenize import word_tokenize, sent_tokenize
from nltk import pos_tag
from nltk.chunk import RegexpParser

from sklearn.preprocessing import OneHotEncoder
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer

# Resurse NLTK (rulează o dată)
nltk.download("punkt", quiet=True)
nltk.download("punkt_tab", quiet=True)
nltk.download("stopwords", quiet=True)
nltk.download("wordnet", quiet=True)
nltk.download("omw-1.4", quiet=True)
nltk.download("averaged_perceptron_tagger", quiet=True)
nltk.download("averaged_perceptron_tagger_eng", quiet=True)

print("OK – librării și resurse încărcate.")

OK – librării și resurse încărcate.


---
# Partea I — Reprezentări cu pandas & scikit-learn

## 1. One-Hot Encoding

**Idee:** fiecare categorie devine o coloană binară (0/1).

| | pandas `get_dummies` | sklearn `OneHotEncoder` |
|---|---|---|
| Ușor de folosit | ✅ foarte simplu | un pic mai verbose |
| Pipeline ML | ❌ mai greu | ✅ se integrează bine |
| Date noi (transform) | trebuie re-aliniat manual | ✅ `transform` pe aceleași categorii |

### 1.1 Dataset de exemplu

In [2]:
df = pd.DataFrame({
    "culoare": ["rosu", "albastru", "verde", "rosu", "verde"],
    "marime": ["S", "M", "L", "M", "S"],
})
df

,culoare,marime
0,rosu,S
1,albastru,M
2,verde,L
3,rosu,M
4,verde,S


### 1.2 pandas — `pd.get_dummies`

In [3]:
# Pas 1: one-hot pe o singură coloană
pd.get_dummies(df["culoare"], prefix="culoare")

,culoare_albastru,culoare_rosu,culoare_verde
0,False,True,False
1,True,False,False
2,False,False,True
3,False,True,False
4,False,False,True


In [4]:
# Pas 2: one-hot pe tot DataFrame-ul (doar coloane categorice)
df_ohe_pandas = pd.get_dummies(df, columns=["culoare", "marime"])
df_ohe_pandas

,culoare_albastru,culoare_rosu,culoare_verde,marime_L,marime_M,marime_S
0,False,True,False,False,False,True
1,True,False,False,False,True,False
2,False,False,True,True,False,False
3,False,True,False,False,True,False
4,False,False,True,False,False,True


In [5]:
# Tip: drop_first=True evită colinearitatea (utile în regresii liniare)
pd.get_dummies(df, columns=["culoare"], drop_first=True)

,marime,culoare_rosu,culoare_verde
0,S,True,False
1,M,False,False
2,L,False,True
3,M,True,False
4,S,False,True


In [38]:
pd.get_dummies(df, columns=["culoare"], drop_first=True, dtype=int)

,marime,culoare_rosu,culoare_verde
0,S,1,0
1,M,0,0
2,L,0,1
3,M,1,0
4,S,0,1


### 1.3 scikit-learn — `OneHotEncoder`

In [6]:
# Pas 1: creăm encoderul
# sparse_output=False → matrice densă (ușor de citit la curs)
encoder = OneHotEncoder(sparse_output=False, handle_unknown="ignore")

# Pas 2: fit pe datele de antrenare (învață categoriile)
X = df[["culoare", "marime"]]
encoder.fit(X)

print("Categorii învățate:")
print(encoder.categories_)

Categorii învățate:
[array(['albastru', 'rosu', 'verde'], dtype=object), array(['L', 'M', 'S'], dtype=object)]


In [7]:
# Pas 3: transform → matrice one-hot
X_ohe = encoder.transform(X)

# Pas 4: nume de coloane lizibile
coloane = encoder.get_feature_names_out(["culoare", "marime"])
df_ohe_sklearn = pd.DataFrame(X_ohe, columns=coloane, dtype=int)
df_ohe_sklearn

,culoare_albastru,culoare_rosu,culoare_verde,marime_L,marime_M,marime_S
0,0,1,0,0,0,1
1,1,0,0,0,1,0
2,0,0,1,1,0,0
3,0,1,0,0,1,0
4,0,0,1,0,0,1


In [8]:
# Avantaj sklearn: transform pe date NOI (categorii necunoscute → 0, nu crash)
date_noi = pd.DataFrame({
    "culoare": ["rosu", "galben"],  # "galben" nu exista la fit
    "marime": ["L", "XL"],          # "XL" nu exista la fit
})
pd.DataFrame(
    encoder.transform(date_noi),
    columns=coloane,
    dtype=int,
)

,culoare_albastru,culoare_rosu,culoare_verde,marime_L,marime_M,marime_S
0,0,1,0,1,0,0
1,0,0,0,0,0,0


**Reține:** `get_dummies` = rapid pentru EDA / notebook-uri. `OneHotEncoder` = preferat în pipeline-uri ML (fit pe train, transform pe test).

## 2. Bag of Words (BoW)

**Idee:** documentul = vector de frecvențe ale cuvintelor (ordinea se ignoră).

Corpus mic de exemplu:

In [9]:
corpus = [
    "pisica mananca peste",
    "cainele mananca carne",
    "pisica si cainele se joaca",
]
corpus

['pisica mananca peste', 'cainele mananca carne', 'pisica si cainele se joaca']

### 2.1 BoW cu `CountVectorizer` (sklearn)

In [10]:
# Pas 1: creăm vectorizerul
bow = CountVectorizer()

# Pas 2: fit_transform pe corpus
X_bow = bow.fit_transform(corpus)

# Pas 3: vocabularul (cuvânt → index coloană)
print("Vocabular:", bow.get_feature_names_out().tolist())
print("Mapping:", bow.vocabulary_)

Vocabular: ['cainele', 'carne', 'joaca', 'mananca', 'peste', 'pisica', 'se', 'si']
Mapping: {'pisica': 5, 'mananca': 3, 'peste': 4, 'cainele': 0, 'carne': 1, 'si': 7, 'se': 6, 'joaca': 2}


In [11]:
# Pas 4: matricea document × termen (frecvențe)
df_bow = pd.DataFrame(
    X_bow.toarray(),
    columns=bow.get_feature_names_out(),
    index=[f"doc{i}" for i in range(len(corpus))],
)
df_bow

,cainele,carne,joaca,mananca,peste,pisica,se,si
doc0,0,0,0,1,1,1,0,0
doc1,1,1,0,1,0,0,0,0
doc2,1,0,1,0,0,1,1,1


In [12]:
# Opțiuni utile: binary=True (prezență 0/1), lowercase, min_df, ngram_range
bow_bin = CountVectorizer(binary=True)
pd.DataFrame(
    bow_bin.fit_transform(corpus).toarray(),
    columns=bow_bin.get_feature_names_out(),
    index=[f"doc{i}" for i in range(len(corpus))],
)

,cainele,carne,joaca,mananca,peste,pisica,se,si
doc0,0,0,0,1,1,1,0,0
doc1,1,1,0,1,0,0,0,0
doc2,1,0,1,0,0,1,1,1


### 2.2 BoW „de mână” (pentru intuiție)

In [13]:
# Pas 1: tokenizare simplă + vocabular sortat
docs_tokens = [doc.lower().split() for doc in corpus]
vocab = sorted({w for doc in docs_tokens for w in doc})
print("Vocabular:", vocab)

# Pas 2: numărăm aparițiile per document
matrice = []
for tokens in docs_tokens:
    counts = Counter(tokens)
    matrice.append([counts[w] for w in vocab])

pd.DataFrame(matrice, columns=vocab, index=[f"doc{i}" for i in range(len(corpus))])

Vocabular: ['cainele', 'carne', 'joaca', 'mananca', 'peste', 'pisica', 'se', 'si']


,cainele,carne,joaca,mananca,peste,pisica,se,si
doc0,0,0,0,1,1,1,0,0
doc1,1,1,0,1,0,0,0,0
doc2,1,0,1,0,0,1,1,1


## 3. TF-IDF (Term Frequency – Inverse Document Frequency)

**Idee:**
- **TF** – cât de des apare un cuvânt *în document*
- **IDF** – cât de rar este cuvântul *în corpus* (cuvintele comune primesc greutate mică)

Formula intuitivă: `TF-IDF = TF × IDF`

### 3.1 TF-IDF cu `TfidfVectorizer` (sklearn)

In [14]:
tfidf = TfidfVectorizer()
X_tfidf = tfidf.fit_transform(corpus)

df_tfidf = pd.DataFrame(
    X_tfidf.toarray().round(3),
    columns=tfidf.get_feature_names_out(),
    index=[f"doc{i}" for i in range(len(corpus))],
)
df_tfidf

,cainele,carne,joaca,mananca,peste,pisica,se,si
doc0,0.000,0.000,0.00,0.518,0.681,0.518,0.00,0.00
doc1,0.518,0.681,0.00,0.518,0.000,0.000,0.00,0.00
doc2,0.373,0.000,0.49,0.000,0.000,0.373,0.49,0.49


In [15]:
# Compară BoW vs TF-IDF pentru același corpus
print("BoW (frecvențe brute):")
display(df_bow)
print("TF-IDF (ponderat):")
display(df_tfidf)

BoW (frecvențe brute):


,cainele,carne,joaca,mananca,peste,pisica,se,si
doc0,0,0,0,1,1,1,0,0
doc1,1,1,0,1,0,0,0,0
doc2,1,0,1,0,0,1,1,1


TF-IDF (ponderat):


,cainele,carne,joaca,mananca,peste,pisica,se,si
doc0,0.000,0.000,0.00,0.518,0.681,0.518,0.00,0.00
doc1,0.518,0.681,0.00,0.518,0.000,0.000,0.00,0.00
doc2,0.373,0.000,0.49,0.000,0.000,0.373,0.49,0.49


In [16]:
# Opțiuni: n-grame, stop words, max_features
tfidf_ngram = TfidfVectorizer(ngram_range=(1, 2), max_features=10)
pd.DataFrame(
    tfidf_ngram.fit_transform(corpus).toarray().round(3),
    columns=tfidf_ngram.get_feature_names_out(),
    index=[f"doc{i}" for i in range(len(corpus))],
)

,cainele,cainele mananca,cainele se,carne,joaca,mananca,mananca carne,mananca peste,peste,pisica
doc0,0.000,0.00,0.000,0.00,0.000,0.428,0.00,0.563,0.563,0.428
doc1,0.373,0.49,0.000,0.49,0.000,0.373,0.49,0.000,0.000,0.000
doc2,0.428,0.00,0.563,0.00,0.563,0.000,0.00,0.000,0.000,0.428


**Reține:** BoW = „câte apariții”. TF-IDF = „cât de important / distinctive e cuvântul în document față de corpus”.

---
# Partea II — Preprocesare: Python clasic vs NLTK

Text de lucru (folosit în toate exemplele de mai jos):

In [17]:
text = (
    "NLP is FUN!!! Students are studying Natural Language Processing. "
    "They studied, they study, and they will study again."
)
text

'NLP is FUN!!! Students are studying Natural Language Processing. They studied, they study, and they will study again.'

## 4. Tokenization

**Idee:** împărțim textul în unități (cuvinte / propoziții).

### 4.1 Python clasic

In [18]:
# Varianta 1: split pe spațiu (naiv – păstrează semnele de punctuație lipite)
tokens_split = text.split()
print("split():", tokens_split)

# Varianta 2: regex – doar secvențe de litere/cifre
tokens_re = re.findall(r"\w+", text.lower())
print("regex:  ", tokens_re)

split(): ['NLP', 'is', 'FUN!!!', 'Students', 'are', 'studying', 'Natural', 'Language', 'Processing.', 'They', 'studied,', 'they', 'study,', 'and', 'they', 'will', 'study', 'again.']
regex:   ['nlp', 'is', 'fun', 'students', 'are', 'studying', 'natural', 'language', 'processing', 'they', 'studied', 'they', 'study', 'and', 'they', 'will', 'study', 'again']


### 4.2 NLTK

In [19]:
# Tokenizare pe cuvinte
tokens_nltk = word_tokenize(text)
print("word_tokenize:", tokens_nltk)

# Tokenizare pe propoziții
sents = sent_tokenize(text)
print("sent_tokenize:", sents)

word_tokenize: ['NLP', 'is', 'FUN', '!', '!', '!', 'Students', 'are', 'studying', 'Natural', 'Language', 'Processing', '.', 'They', 'studied', ',', 'they', 'study', ',', 'and', 'they', 'will', 'study', 'again', '.']
sent_tokenize: ['NLP is FUN!!!', 'Students are studying Natural Language Processing.', 'They studied, they study, and they will study again.']


## 5. Punctuation Removal

**Idee:** eliminăm `. , ! ? " '` etc.

### 5.1 Python clasic

In [20]:
# Metoda 1: translate + string.punctuation
text_no_punct = text.translate(str.maketrans("", "", string.punctuation))
print("translate:", text_no_punct)

# Metoda 2: regex – păstrăm litere, cifre, spații
text_no_punct_re = re.sub(r"[^\w\s]", "", text)
print("regex:    ", text_no_punct_re)

translate: NLP is FUN Students are studying Natural Language Processing They studied they study and they will study again
regex:     NLP is FUN Students are studying Natural Language Processing They studied they study and they will study again


### 5.2 NLTK (+ filtrare pe tokeni)

In [21]:
tokens = word_tokenize(text)
# Păstrăm doar tokenii care NU sunt punctuație pură
tokens_no_punct = [t for t in tokens if t not in string.punctuation]
print(tokens_no_punct)

['NLP', 'is', 'FUN', 'Students', 'are', 'studying', 'Natural', 'Language', 'Processing', 'They', 'studied', 'they', 'study', 'and', 'they', 'will', 'study', 'again']


## 6. Stopword Removal

**Idee:** eliminăm cuvinte foarte frecvente, cu puțină informație (`the`, `is`, `and`…).

### 6.1 Python clasic (listă proprie)

In [22]:
my_stopwords = {"is", "are", "and", "they", "will", "a", "the", "of"}

tokens = re.findall(r"\w+", text.lower())
tokens_no_sw = [t for t in tokens if t not in my_stopwords]
print("Înainte:", tokens)
print("După:   ", tokens_no_sw)

Înainte: ['nlp', 'is', 'fun', 'students', 'are', 'studying', 'natural', 'language', 'processing', 'they', 'studied', 'they', 'study', 'and', 'they', 'will', 'study', 'again']
După:    ['nlp', 'fun', 'students', 'studying', 'natural', 'language', 'processing', 'studied', 'study', 'study', 'again']


### 6.2 NLTK (`stopwords`)

In [23]:
en_stop = set(stopwords.words("english"))

tokens = word_tokenize(text.lower())
tokens = [t for t in tokens if t not in string.punctuation]
tokens_no_sw_nltk = [t for t in tokens if t not in en_stop]

print("Nr. stopwords EN:", len(en_stop))
print("După filtrare:   ", tokens_no_sw_nltk)

Nr. stopwords EN: 198
După filtrare:    ['nlp', 'fun', 'students', 'studying', 'natural', 'language', 'processing', 'studied', 'study', 'study']


## 7. Stemming

**Idee:** taiem sufixe după reguli heuristice → rădăcină aproximativă (`studying` → `studi`).
Rapid, dar poate produce forme care nu sunt cuvinte reale.

### 7.1 Python clasic (reguli naive)

In [24]:
def simple_stem(word: str) -> str:
    """Stemmer didactic – NU e echivalent cu Porter!"""
    for suf in ("ing", "ed", "ies", "ly", "es", "s"):
        if word.endswith(suf) and len(word) - len(suf) >= 3:
            return word[: -len(suf)]
    return word

cuvinte = ["studying", "studied", "studies", "cats", "running", "happily"]
for w in cuvinte:
    print(f"{w:12} → {simple_stem(w)}")

studying     → study
studied      → studi
studies      → stud
cats         → cat
running      → runn
happily      → happi


### 7.2 NLTK — `PorterStemmer`

In [25]:
stemmer = PorterStemmer()

cuvinte = ["studying", "studied", "studies", "cats", "running", "happily", "better"]
for w in cuvinte:
    print(f"{w:12} → {stemmer.stem(w)}")

studying     → studi
studied      → studi
studies      → studi
cats         → cat
running      → run
happily      → happili
better       → better


## 8. Lemmatization

**Idee:** reducem cuvântul la forma de dicționar (lema), folosind morfologie (`studied` → `study`, `better` → `good`).
Mai lent decât stemming, dar rezultatul e un cuvânt real.

### 8.1 Python clasic (dicționar mic de excepții – didactic)

In [26]:
LEMMA_MAP = {
    "studied": "study",
    "studying": "study",
    "studies": "study",
    "are": "be",
    "is": "be",
    "was": "be",
    "better": "good",
    "cats": "cat",
}

def simple_lemma(word: str) -> str:
    return LEMMA_MAP.get(word.lower(), word.lower())

for w in ["studied", "studying", "better", "cats", "NLP"]:
    print(f"{w:12} → {simple_lemma(w)}")

studied      → study
studying     → study
better       → good
cats         → cat
NLP          → nlp


### 8.2 NLTK — `WordNetLemmatizer`

> Important: lemmatizer-ul funcționează mai bine dacă îi spui **POS-ul** (`v`=verb, `n`=noun, `a`=adj).

In [27]:
lemmatizer = WordNetLemmatizer()

exemple = [
    ("studied", "v"),
    ("studying", "v"),
    ("studies", "n"),
    ("better", "a"),
    ("cats", "n"),
    ("running", "v"),
    ("running", "n"),  # același cuvânt, POS diferit
]

for word, pos in exemple:
    print(f"{word:12} (pos={pos}) → {lemmatizer.lemmatize(word, pos=pos)}")

studied      (pos=v) → study
studying     (pos=v) → study
studies      (pos=n) → study
better       (pos=a) → good
cats         (pos=n) → cat
running      (pos=v) → run
running      (pos=n) → running


In [28]:
# Stemming vs Lemmatization – side by side
print(f"{'cuvânt':12} {'stem':12} {'lemma(v)'}")
print("-" * 40)
for w in ["studying", "studied", "better", "mice", "running"]:
    print(f"{w:12} {stemmer.stem(w):12} {lemmatizer.lemmatize(w, pos='v')}")

cuvânt       stem         lemma(v)
----------------------------------------
studying     studi        study
studied      studi        study
better       better       better
mice         mice         mice
running      run          run


## 9. Text Normalization

**Idee:** aducem textul la o formă consistentă înainte de modelare.
Pași tipici: lowercase → fără punctuație → fără stopwords → lemmatize/stem.

### 9.1 Python clasic (pipeline scurt)

In [29]:
def normalize_classic(text: str) -> list[str]:
    # 1) lowercase
    t = text.lower()
    # 2) scoatem punctuația
    t = re.sub(r"[^\w\s]", " ", t)
    # 3) tokenizare
    tokens = t.split()
    # 4) stopwords proprii
    stop = {"is", "are", "and", "they", "will", "a", "the", "of"}
    tokens = [w for w in tokens if w not in stop]
    # 5) stemming simplu
    tokens = [simple_stem(w) for w in tokens]
    return tokens

print("Original:", text)
print("Normalizat:", normalize_classic(text))

Original: NLP is FUN!!! Students are studying Natural Language Processing. They studied, they study, and they will study again.
Normalizat: ['nlp', 'fun', 'student', 'study', 'natural', 'language', 'process', 'studi', 'study', 'study', 'again']


### 9.2 NLTK (pipeline)

In [30]:
def normalize_nltk(text: str) -> list[str]:
    # 1) lowercase + tokenizare
    tokens = word_tokenize(text.lower())
    # 2) fără punctuație
    tokens = [t for t in tokens if t not in string.punctuation]
    # 3) fără stopwords
    stop = set(stopwords.words("english"))
    tokens = [t for t in tokens if t not in stop]
    # 4) lemmatizare (ca substantiv by default; pentru producție: POS pe fiecare token)
    tokens = [lemmatizer.lemmatize(t) for t in tokens]
    return tokens

print("Original:", text)
print("Normalizat:", normalize_nltk(text))

Original: NLP is FUN!!! Students are studying Natural Language Processing. They studied, they study, and they will study again.
Normalizat: ['nlp', 'fun', 'student', 'studying', 'natural', 'language', 'processing', 'studied', 'study', 'study']


## 10. Parts of Speech (POS) Tagging

**Idee:** etichetăm fiecare cuvânt cu rolul gramatical (NN=noun, VB=verb, JJ=adjective…).

### 10.1 Python clasic (reguli foarte simple – didactic)

In [31]:
def simple_pos_tag(tokens: list[str]) -> list[tuple[str, str]]:
    """Tagger pe reguli – doar pentru intuiție."""
    tagged = []
    for t in tokens:
        tl = t.lower()
        if tl in {"the", "a", "an"}:
            tag = "DT"          # determiner
        elif tl in {"is", "are", "was", "study", "studied", "studying"}:
            tag = "VERB"
        elif t[0].isupper() and t.isalpha():
            tag = "NNP"         # proper noun (heuristica capitalizării)
        elif tl.endswith("ly"):
            tag = "ADV"
        elif tl.endswith("ing"):
            tag = "VBG"
        else:
            tag = "NOUN"        # default naiv
        tagged.append((t, tag))
    return tagged

toks = re.findall(r"\w+", "NLP is fun and students are studying")
simple_pos_tag(toks)

[('NLP', 'NNP'),
 ('is', 'VERB'),
 ('fun', 'NOUN'),
 ('and', 'NOUN'),
 ('students', 'NOUN'),
 ('are', 'VERB'),
 ('studying', 'VERB')]

### 10.2 NLTK — `pos_tag`

In [32]:
propozitie = "NLP students are studying natural language processing."
tokens = word_tokenize(propozitie)
tagged = pos_tag(tokens)

print("Token → POS")
for word, tag in tagged:
    print(f"  {word:15} {tag}")

Token → POS
  NLP             JJ
  students        NNS
  are             VBP
  studying        VBG
  natural         JJ
  language        NN
  processing      NN
  .               .


In [33]:
# Cheat-sheet scurt pentru tag-uri frecvente (Penn Treebank)
penn = {
    "NN": "noun, singular",
    "NNS": "noun, plural",
    "NNP": "proper noun",
    "VB": "verb, base",
    "VBD": "verb, past",
    "VBG": "verb, gerund/ing",
    "JJ": "adjective",
    "RB": "adverb",
    "DT": "determiner (the/a)",
    "IN": "preposition",
    "PRP": "pronoun",
}
pd.DataFrame(list(penn.items()), columns=["tag", "semnificație"])

,tag,semnificație
0,NN,"noun, singular"
1,NNS,"noun, plural"
2,NNP,proper noun
3,VB,"verb, base"
4,VBD,"verb, past"
5,VBG,"verb, gerund/ing"
6,JJ,adjective
7,RB,adverb
8,DT,determiner (the/a)
9,IN,preposition


## 11. Parsing

**Idee:** descoperim structura sintactică (fragmente: NP = noun phrase, VP = verb phrase).
Aici folosim **chunking** cu reguli (RegexpParser) – varianta didactică din NLTK.

### 11.1 Python clasic (grupare naivă NP)

In [34]:
def naive_np_chunks(tagged: list[tuple[str, str]]) -> list[str]:
    """Grupează secvențe de tip (DT)? (JJ)* (NOUN)+ → NP."""
    chunks, i = [], 0
    while i < len(tagged):
        word, tag = tagged[i]
        if tag in {"DT", "JJ", "NOUN", "NNP", "NN", "NNS"}:
            phrase = [word]
            i += 1
            while i < len(tagged) and tagged[i][1] in {"JJ", "NOUN", "NNP", "NN", "NNS"}:
                phrase.append(tagged[i][0])
                i += 1
            chunks.append("NP: " + " ".join(phrase))
        else:
            chunks.append(f"{tag}: {word}")
            i += 1
    return chunks

tagged_simple = simple_pos_tag(re.findall(r"\w+", "The quick brown fox jumps"))
print("POS:", tagged_simple)
print("Chunks:")
for c in naive_np_chunks(tagged_simple):
    print(" ", c)

POS: [('The', 'DT'), ('quick', 'NOUN'), ('brown', 'NOUN'), ('fox', 'NOUN'), ('jumps', 'NOUN')]
Chunks:
  NP: The quick brown fox jumps


### 11.2 NLTK — `RegexpParser` (chunk grammar)

In [35]:
propozitie = "The quick brown fox jumps over the lazy dog"
tokens = word_tokenize(propozitie)
tagged = pos_tag(tokens)

# Gramatică: NP = (determiner)? + adjective* + noun+
grammar = r"""
NP: {<DT>?<JJ>*<NN.*>+}
"""

parser = RegexpParser(grammar)
tree = parser.parse(tagged)

print(tree)
print()
print("Doar chunk-urile NP:")
for subtree in tree.subtrees(filter=lambda t: t.label() == "NP"):
    print(" -", " ".join(word for word, tag in subtree.leaves()))

(S
  (NP The/DT quick/JJ brown/NN fox/NN)
  jumps/VBZ
  over/IN
  (NP the/DT lazy/JJ dog/NN))

Doar chunk-urile NP:
 - The quick brown fox
 - the lazy dog


In [36]:
# Gramatică un pic mai bogată: NP + VP
grammar2 = r"""
NP: {<DT>?<JJ>*<NN.*>+}
PP: {<IN><NP>}
VP: {<VB.*><NP|PP>*}
"""
tree2 = RegexpParser(grammar2).parse(tagged)
print(tree2)

(S
  (NP The/DT quick/JJ brown/NN fox/NN)
  (VP jumps/VBZ (PP over/IN (NP the/DT lazy/JJ dog/NN))))


---
# Recapitulare rapidă

| Tehnică | Ce face | Tool tipic |
|---|---|---|
| One-Hot | categorii → coloane 0/1 | `pd.get_dummies` / `OneHotEncoder` |
| BoW | document → frecvențe cuvinte | `CountVectorizer` |
| TF-IDF | BoW ponderat cu raritatea în corpus | `TfidfVectorizer` |
| Tokenization | text → tokeni | `split` / `word_tokenize` |
| Punctuation removal | scoate semnele | `string.punctuation` / filtrare tokeni |
| Stopword removal | scoate cuvinte „goale” | listă proprie / `stopwords` |
| Stemming | taie sufixe (rapid, aproximativ) | reguli / `PorterStemmer` |
| Lemmatization | formă de dicționar (mai corect) | map / `WordNetLemmatizer` |
| Normalization | pipeline de curățare | funcție custom |
| POS tagging | rol gramatical | reguli / `pos_tag` |
| Parsing (chunking) | fragmente sintactice NP/VP | reguli / `RegexpParser` |

**Ordine tipică de preprocesare:**  
`lowercase → tokenize → remove punct → remove stopwords → stem/lemmatize → BoW/TF-IDF`